# P10.6-AI — Notebook 54: split interno y plan del primer clasificador

Este notebook toma el dataset RSNA LumbarDISC validado por el Notebook 53 y:

- construye un split reproducible por `study_id`;
- evita fuga de estudios entre entrenamiento, validación y prueba interna;
- prepara manifests para **spinal canal stenosis** con secuencias **Sagittal T2/STIR**;
- documenta el plan del primer modelo 2.5D;
- guarda únicamente artefactos sanitizados en Google Drive.

No entrena, no accede al test oficial y no genera diagnóstico clínico.


## Guardias

- `humanReviewRequired=true`
- `notClinicalDiagnosis=true`
- El split se realiza por `study_id`, nunca por imagen, serie o corte.
- El test oficial de Kaggle no se usa.
- Los DICOM permanecen fuera de Git.
- El primer modelo planificado es `rsna_central_stenosis_sagittal_t2_2p5d.pt`.


In [ ]:
# 1) Dependencias e imports
from __future__ import annotations

import hashlib
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
# 2) Montar Drive y resolver rutas
from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")

# Usar primero la copia local si sigue disponible.
# Si Colab reinició el runtime, usar la copia permanente de Google Drive.
LOCAL_RSNA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
DRIVE_RSNA_ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC"
)

if LOCAL_RSNA_ROOT.exists():
    RSNA_ROOT = LOCAL_RSNA_ROOT
    DATA_SOURCE = "local_runtime"
elif DRIVE_RSNA_ROOT.exists():
    RSNA_ROOT = DRIVE_RSNA_ROOT
    DATA_SOURCE = "google_drive"
else:
    raise RuntimeError(
        "No se encontró RSNA ni en /content ni en Google Drive. "
        "Ejecutá primero el Notebook 53."
    )

OUTPUT_ROOT = (
    PFI_ROOT
    / "results"
    / "P10_6_rsna_findings"
    / "notebook54_split"
)
MODEL_ROOT = (
    PFI_ROOT
    / "models"
    / "P10_6_rsna_findings"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = RSNA_ROOT / "train.csv"
SERIES_CSV = RSNA_ROOT / "train_series_descriptions.csv"
COORDINATES_CSV = RSNA_ROOT / "train_label_coordinates.csv"
TRAIN_IMAGES = RSNA_ROOT / "train_images"

required = [
    TRAIN_CSV,
    SERIES_CSV,
    COORDINATES_CSV,
    TRAIN_IMAGES,
]
missing = [str(path) for path in required if not path.exists()]

if missing:
    raise RuntimeError(
        "Falta el dataset validado por Notebook 53.\n- "
        + "\n- ".join(missing)
    )

print({
    "dataSource": DATA_SOURCE,
    "rsnaRoot": str(RSNA_ROOT),
    "outputRoot": str(OUTPUT_ROOT),
    "modelRoot": str(MODEL_ROOT),
})


In [ ]:
# 3) Configuración reproducible
SEED = 2026
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
INTERNAL_TEST_FRACTION = 0.15

random.seed(SEED)
np.random.seed(SEED)

SEVERITY_ORDER = {
    "Normal/Mild": 0,
    "Moderate": 1,
    "Severe": 2,
}
LEVELS = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]
TARGET_CONDITION = "spinal_canal_stenosis"
TARGET_SEQUENCE = "Sagittal T2/STIR"
FINAL_MODEL_NAME = "rsna_central_stenosis_sagittal_t2_2p5d.pt"

assert abs(
    TRAIN_FRACTION
    + VALIDATION_FRACTION
    + INTERNAL_TEST_FRACTION
    - 1.0
) < 1e-9


In [ ]:
# 4) Cargar tablas oficiales de entrenamiento
train_wide = pd.read_csv(TRAIN_CSV)
series = pd.read_csv(SERIES_CSV)
coordinates = pd.read_csv(COORDINATES_CSV)

for frame, name in [
    (train_wide, "train.csv"),
    (series, "train_series_descriptions.csv"),
    (coordinates, "train_label_coordinates.csv"),
]:
    if "study_id" not in frame.columns:
        raise RuntimeError(f"{name} no contiene study_id")

train_wide["study_id"] = train_wide["study_id"].astype(str)
series["study_id"] = series["study_id"].astype(str)
series["series_id"] = series["series_id"].astype(str)
coordinates["study_id"] = coordinates["study_id"].astype(str)
coordinates["series_id"] = coordinates["series_id"].astype(str)

print({
    "nStudies": int(train_wide["study_id"].nunique()),
    "nSeries": int(series["series_id"].nunique()),
    "nCoordinateRows": int(len(coordinates)),
})


In [ ]:
# 5) Construir estrato por severidad máxima de estenosis central
target_columns = [
    column
    for column in train_wide.columns
    if column.startswith("spinal_canal_stenosis_")
]

if len(target_columns) != 5:
    raise RuntimeError(
        f"Se esperaban 5 columnas y se encontraron "
        f"{len(target_columns)}: {target_columns}"
    )


def normalize_severity(value: object) -> str:
    if pd.isna(value):
        return "Missing"
    return str(value).strip()


severity_numeric = train_wide[target_columns].apply(
    lambda col: col.map(
        lambda value: SEVERITY_ORDER.get(
            normalize_severity(value),
            -1,
        )
    )
)

train_wide["central_max_severity_code"] = (
    severity_numeric.max(axis=1)
)
train_wide["central_max_severity"] = (
    train_wide["central_max_severity_code"].map({
        -1: "Missing",
        0: "Normal/Mild",
        1: "Moderate",
        2: "Severe",
    })
)

study_table = train_wide[
    [
        "study_id",
        "central_max_severity",
        "central_max_severity_code",
    ]
] .copy()

print(
    study_table["central_max_severity"]
    .value_counts(dropna=False)
    .to_dict()
)


In [ ]:
# 6) Split determinista 70/15/15 por study_id y severidad máxima
#
# Los estratos con menos de 3 estudios no pueden distribuirse de forma
# válida entre train, validation e internal_test. Se conservan en train.

stratum_counts = (
    study_table["central_max_severity"]
    .value_counts(dropna=False)
)

rare_strata = set(
    stratum_counts[
        stratum_counts < 3
    ].index.tolist()
)

rare_studies = study_table[
    study_table["central_max_severity"].isin(rare_strata)
] .copy()

stratifiable_studies = study_table[
    ~study_table["central_max_severity"].isin(rare_strata)
] .copy()

print({
    "stratumCounts": stratum_counts.to_dict(),
    "rareStrataAssignedToTrain": sorted(
        map(str, rare_strata)
    ),
    "rareStudyCount": int(len(rare_studies)),
    "stratifiableStudyCount": int(
        len(stratifiable_studies)
    ),
})

train_regular, holdout_studies = train_test_split(
    stratifiable_studies,
    test_size=(
        VALIDATION_FRACTION
        + INTERNAL_TEST_FRACTION
    ),
    random_state=SEED,
    stratify=stratifiable_studies[
        "central_max_severity"
    ],
)

relative_test_fraction = (
    INTERNAL_TEST_FRACTION
    / (
        VALIDATION_FRACTION
        + INTERNAL_TEST_FRACTION
    )
)

validation_studies, test_studies = train_test_split(
    holdout_studies,
    test_size=relative_test_fraction,
    random_state=SEED,
    stratify=holdout_studies[
        "central_max_severity"
    ],
)

train_studies = pd.concat(
    [train_regular, rare_studies],
    ignore_index=True,
)

train_studies = train_studies.assign(split="train")
validation_studies = validation_studies.assign(
    split="validation"
)
test_studies = test_studies.assign(
    split="internal_test"
)

study_splits = (
    pd.concat(
        [
            train_studies,
            validation_studies,
            test_studies,
        ],
        ignore_index=True,
    )
    .sort_values(["split", "study_id"])
    .reset_index(drop=True)
)

original_ids = set(study_table["study_id"])
split_ids = set(study_splits["study_id"])

if original_ids != split_ids:
    raise RuntimeError(
        "El split no conserva exactamente los estudios."
    )

if study_splits["study_id"].duplicated().any():
    raise RuntimeError(
        "Se detectó fuga: un study_id aparece "
        "más de una vez."
    )

print(
    pd.crosstab(
        study_splits["split"],
        study_splits["central_max_severity"],
        margins=True,
    )
)

print({
    "trainStudies": int(
        (study_splits["split"] == "train").sum()
    ),
    "validationStudies": int(
        (study_splits["split"] == "validation").sum()
    ),
    "internalTestStudies": int(
        (
            study_splits["split"]
            == "internal_test"
        ).sum()
    ),
    "totalStudies": int(len(study_splits)),
    "studyLeakage": False,
})


In [ ]:
# 7) Pasar etiquetas wide a formato largo por nivel
#
# Las etiquetas Missing se registran y se excluyen del manifest de entrenamiento.
# No se convierten en una cuarta clase.

level_suffix_to_label = {
    "l1_l2": "L1/L2",
    "l2_l3": "L2/L3",
    "l3_l4": "L3/L4",
    "l4_l5": "L4/L5",
    "l5_s1": "L5/S1",
}

long_parts = []

for column in target_columns:
    suffix = column.removeprefix(
        "spinal_canal_stenosis_"
    )
    level = level_suffix_to_label.get(suffix)

    if level is None:
        raise RuntimeError(
            f"Nivel no reconocido: {column}"
        )

    part = (
        train_wide[["study_id", column]]
        .rename(columns={column: "severity"})
        .copy()
    )
    part["level"] = level
    long_parts.append(part)

central_labels_all = pd.concat(
    long_parts,
    ignore_index=True,
)

central_labels_all["severity"] = (
    central_labels_all["severity"]
    .map(normalize_severity)
)

missing_labels = central_labels_all[
    central_labels_all["severity"].eq("Missing")
] .copy()

print({
    "totalLevelLabels": int(
        len(central_labels_all)
    ),
    "missingLevelLabels": int(
        len(missing_labels)
    ),
    "studiesWithMissingLabels": int(
        missing_labels["study_id"].nunique()
    ),
})

central_labels = central_labels_all[
    ~central_labels_all["severity"].eq("Missing")
] .copy()

central_labels["severity_code"] = (
    central_labels["severity"]
    .map(SEVERITY_ORDER)
)

if central_labels["severity_code"].isna().any():
    unknown = sorted(
        central_labels.loc[
            central_labels["severity_code"].isna(),
            "severity",
        ]
        .dropna()
        .unique()
        .tolist()
    )
    raise RuntimeError(
        f"Severidades no reconocidas: {unknown}"
    )

central_labels["severity_code"] = (
    central_labels["severity_code"]
    .astype(int)
)

central_labels = central_labels.merge(
    study_splits[["study_id", "split"]],
    on="study_id",
    how="left",
    validate="many_to_one",
)

if central_labels["split"].isna().any():
    raise RuntimeError(
        "Hay estudios etiquetados sin split asignado."
    )

print({
    "usableLevelLabels": int(
        len(central_labels)
    ),
    "excludedMissingLabels": int(
        len(missing_labels)
    ),
    "usableStudies": int(
        central_labels["study_id"].nunique()
    ),
    "severityDistribution": (
        central_labels["severity"]
        .value_counts()
        .to_dict()
    ),
})


In [ ]:
# 8) Seleccionar series Sagittal T2/STIR y construir manifest
target_series = series.loc[
    series["series_description"]
    .astype(str)
    .str.strip()
    .eq(TARGET_SEQUENCE)
] .copy()

if target_series.empty:
    raise RuntimeError(
        f"No se encontraron series {TARGET_SEQUENCE!r}"
    )

manifest = central_labels.merge(
    target_series[
        [
            "study_id",
            "series_id",
            "series_description",
        ]
    ],
    on="study_id",
    how="inner",
    validate="many_to_many",
)

manifest["series_path"] = manifest.apply(
    lambda row: str(
        TRAIN_IMAGES
        / row["study_id"]
        / row["series_id"]
    ),
    axis=1,
)
manifest["series_exists"] = (
    manifest["series_path"]
    .map(lambda value: Path(value).is_dir())
)
manifest["condition_key"] = TARGET_CONDITION
manifest["source_plane"] = "sagittal"
manifest["source_sequence"] = TARGET_SEQUENCE
manifest["prediction_source"] = "image_classifier"
manifest["human_review_required"] = True
manifest["not_clinical_diagnosis"] = True

if not manifest["series_exists"].all():
    raise RuntimeError(
        f"Faltan "
        f"{int((~manifest['series_exists']).sum())} "
        f"rutas de series en disco."
    )

manifest = (
    manifest[
        [
            "study_id",
            "split",
            "series_id",
            "series_description",
            "series_path",
            "level",
            "condition_key",
            "severity",
            "severity_code",
            "source_plane",
            "source_sequence",
            "prediction_source",
            "human_review_required",
            "not_clinical_diagnosis",
        ]
    ]
    .sort_values(
        [
            "split",
            "study_id",
            "series_id",
            "level",
        ]
    )
    .reset_index(drop=True)
)

print({
    "manifestRows": int(len(manifest)),
    "manifestStudies": int(
        manifest["study_id"].nunique()
    ),
    "manifestSeries": int(
        manifest["series_id"].nunique()
    ),
})


In [ ]:
# 9) Validaciones de fuga y cobertura
split_sets = {
    split: set(group["study_id"])
    for split, group in study_splits.groupby("split")
}

overlaps = {
    "train_validation": sorted(
        split_sets["train"]
        & split_sets["validation"]
    ),
    "train_internal_test": sorted(
        split_sets["train"]
        & split_sets["internal_test"]
    ),
    "validation_internal_test": sorted(
        split_sets["validation"]
        & split_sets["internal_test"]
    ),
}

if any(overlaps.values()):
    raise RuntimeError(
        f"Se detectó fuga: {overlaps}"
    )

coverage = (
    manifest.groupby("split")
    .agg(
        studies=("study_id", "nunique"),
        series=("series_id", "nunique"),
        rows=("study_id", "size"),
    )
    .reset_index()
)

label_distribution = (
    manifest.groupby(
        ["split", "level", "severity"]
    )
    .size()
    .rename("count")
    .reset_index()
)

print(coverage)
print(label_distribution.head(15))


In [ ]:
# 10) Plan del primer modelo
model_plan = {
    "schemaVersion": "pfi.rsna-model-plan.v1",
    "dataset": "RSNA_LumbarDISC",
    "task": {
        "findingType": "spinal_canal_stenosis",
        "levels": LEVELS,
        "classes": [
            "Normal/Mild",
            "Moderate",
            "Severe",
        ],
        "sourcePlane": "sagittal",
        "sourceSequence": TARGET_SEQUENCE,
    },
    "input": {
        "strategy": "2.5D",
        "neighborSlices": 3,
        "plannedImageSize": [224, 224],
        "normalization": (
            "per-series robust intensity normalization"
        ),
        "seriesSelection": (
            "Sagittal T2/STIR only"
        ),
    },
    "split": {
        "unit": "study_id",
        "seed": SEED,
        "trainFraction": TRAIN_FRACTION,
        "validationFraction": VALIDATION_FRACTION,
        "internalTestFraction": INTERNAL_TEST_FRACTION,
        "stratification": (
            "maximum spinal canal stenosis severity "
            "per study"
        ),
        "rareStrataPolicy": (
            "strata with fewer than 3 studies "
            "remain in train"
        ),
        "missingLabelPolicy": (
            "exclude missing level labels from manifests"
        ),
        "officialTestAccessed": False,
    },
    "training": {
        "notebook": (
            "55_P10_6_RSNA_central_stenosis_training.ipynb"
        ),
        "architectureCandidates": [
            "timm efficientnet_b0",
            "timm convnext_tiny",
        ],
        "lossCandidates": [
            "weighted_cross_entropy",
            "focal_loss",
        ],
        "checkpointSelection": [
            "macro_f1",
            "severe_recall",
            "balanced_accuracy",
        ],
        "mixedPrecision": True,
        "earlyStopping": True,
    },
    "evaluation": {
        "notebook": (
            "56_P10_6_RSNA_central_stenosis_"
            "evaluation_export.ipynb"
        ),
        "metrics": [
            "recall_sensitivity",
            "precision",
            "specificity",
            "macro_f1",
            "balanced_accuracy",
            "roc_auc_one_vs_rest",
            "confusion_matrix",
            "weighted_log_loss",
            "calibration",
            "severe_false_negatives",
            "metrics_by_level",
        ],
        "finalArtifact": FINAL_MODEL_NAME,
    },
    "governance": {
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
        "predictionStatus": "pending_review",
        "measurementsAreSecondaryEvidence": True,
    },
}

print(json.dumps(model_plan, indent=2))


In [ ]:
# 11) Guardar artefactos sanitizados
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


paths = {
    "study_splits.csv": (
        OUTPUT_ROOT / "study_splits.csv"
    ),
    "central_stenosis_sagittal_t2_manifest.csv": (
        OUTPUT_ROOT
        / "central_stenosis_sagittal_t2_manifest.csv"
    ),
    "train_manifest.csv": (
        OUTPUT_ROOT / "train_manifest.csv"
    ),
    "validation_manifest.csv": (
        OUTPUT_ROOT / "validation_manifest.csv"
    ),
    "internal_test_manifest.csv": (
        OUTPUT_ROOT / "internal_test_manifest.csv"
    ),
    "label_distribution_by_split.csv": (
        OUTPUT_ROOT
        / "label_distribution_by_split.csv"
    ),
    "split_coverage.csv": (
        OUTPUT_ROOT / "split_coverage.csv"
    ),
    "model_plan.json": (
        OUTPUT_ROOT / "model_plan.json"
    ),
    "split_summary.json": (
        OUTPUT_ROOT / "split_summary.json"
    ),
}

study_splits.to_csv(
    paths["study_splits.csv"],
    index=False,
)
manifest.to_csv(
    paths[
        "central_stenosis_sagittal_t2_manifest.csv"
    ],
    index=False,
)
manifest.loc[
    manifest["split"].eq("train")
] .to_csv(
    paths["train_manifest.csv"],
    index=False,
)
manifest.loc[
    manifest["split"].eq("validation")
] .to_csv(
    paths["validation_manifest.csv"],
    index=False,
)
manifest.loc[
    manifest["split"].eq("internal_test")
] .to_csv(
    paths["internal_test_manifest.csv"],
    index=False,
)
label_distribution.to_csv(
    paths["label_distribution_by_split.csv"],
    index=False,
)
coverage.to_csv(
    paths["split_coverage.csv"],
    index=False,
)

with paths["model_plan.json"].open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        model_plan,
        handle,
        indent=2,
        ensure_ascii=False,
    )

split_summary = {
    "schemaVersion": "pfi.rsna-split-summary.v1",
    "seed": SEED,
    "dataSource": DATA_SOURCE,
    "studyCounts": (
        study_splits["split"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "manifestStudyCounts": (
        manifest.groupby("split")["study_id"]
        .nunique()
        .to_dict()
    ),
    "manifestSeriesCounts": (
        manifest.groupby("split")["series_id"]
        .nunique()
        .to_dict()
    ),
    "rareStrataAssignedToTrain": sorted(
        map(str, rare_strata)
    ),
    "excludedMissingLevelLabels": int(
        len(missing_labels)
    ),
    "leakageDetected": False,
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
    "sourceHashes": {
        "train.csv": sha256_file(TRAIN_CSV),
        "train_series_descriptions.csv": (
            sha256_file(SERIES_CSV)
        ),
        "train_label_coordinates.csv": (
            sha256_file(COORDINATES_CSV)
        ),
    },
}

with paths["split_summary.json"].open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        split_summary,
        handle,
        indent=2,
        ensure_ascii=False,
    )

print(json.dumps({
    "outputs": {
        name: str(path)
        for name, path in paths.items()
    },
    "outputHashes": {
        name: sha256_file(path)
        for name, path in paths.items()
    },
}, indent=2))


In [ ]:
# 12) Gate para habilitar Notebook 55
required_checks = {
    "allStudiesAssignedOnce": (
        study_splits["study_id"].nunique()
        == train_wide["study_id"].nunique()
        and not study_splits["study_id"]
        .duplicated()
        .any()
    ),
    "noLeakage": not any(overlaps.values()),
    "targetSeriesAvailable": (
        manifest["series_id"].nunique() > 0
    ),
    "allSeriesPathsExist": bool(
        manifest["series_path"]
        .map(lambda value: Path(value).is_dir())
        .all()
    ),
    "allThreeSeveritiesPresentInTrain": (
        set(
            manifest.loc[
                manifest["split"].eq("train"),
                "severity",
            ].unique()
        )
        == set(SEVERITY_ORDER)
    ),
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}

blocking_failures = [
    key
    for key, value in required_checks.items()
    if key != "officialTestAccessed"
    and value is not True
]

if required_checks["officialTestAccessed"] is not False:
    blocking_failures.append(
        "officialTestAccessed"
    )

print(json.dumps(required_checks, indent=2))

if blocking_failures:
    raise RuntimeError(
        "Notebook 55 no habilitado. Fallaron: "
        + ", ".join(blocking_failures)
    )

print({
    "status": "APPROVED_FOR_NOTEBOOK_55",
    "nextNotebook": (
        "55_P10_6_RSNA_central_stenosis_training.ipynb"
    ),
    "plannedFinalArtifact": FINAL_MODEL_NAME,
})
